In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate

import torch

In [2]:
ag = [[4], [2,5], [1,2], [4, 7], [4,5,7], [0,1,4], [7,8], [6,7], [3,4, 6]]
gate_order = [8,5,2,1,4,7,6,3,0]

def load_unitary_gates():
    gates = []
    for i in gate_order:
        # 注意：这里需要把 torch tensor 转为 numpy array 以便后续处理
        gate = torch.load(f'../gates_2patterns/unitary_gates/tensor{i}.pt').detach().numpy()
        gate = gate.reshape(-1, 2**len(ag[i]))
        gates.append(gate)
    return gates

In [3]:
qc = QuantumCircuit(9)
gates = load_unitary_gates()
# gates[0]
apply_list = [ag[i] for i in gate_order]
for gate_matrix, qubits in zip(gates, apply_list):
        # 使用 Qiskit 的 UnitaryGate 包装你的矩阵
        print(gate_matrix.shape, qubits)
        custom_gate = UnitaryGate(gate_matrix)
        qc.append(custom_gate, qubits[::-1])
    # 添加全测量
qc.measure_all()

(8, 8) [3, 4, 6]
(8, 8) [0, 1, 4]
(4, 4) [1, 2]
(4, 4) [2, 5]
(8, 8) [4, 5, 7]
(4, 4) [6, 7]
(4, 4) [7, 8]
(4, 4) [4, 7]
(2, 2) [4]


In [4]:
## 经典模拟器的结果
from qiskit_aer import AerSimulator
from qiskit import transpile

# 选择模拟器
backend = AerSimulator()

# 编译并运行
t_qc = transpile(qc, backend)
job = backend.run(t_qc, shots=4096)
result = job.result()

counts = result.get_counts()
print(counts)

{'010100001': 2015, '001100100': 2081}


In [19]:
import qiskit.qasm2 as qasm2
from qiskit.transpiler import CouplingMap
# 1. 明确你要用的 9 个物理比特
physical_qubits = [25, 31, 38, 30, 37, 43, 36, 42, 49]

# 2. 手动构建 3x3 正方格点的连线关系 (Coupling Map)
# 按照你给的行列关系，定义它们之间的物理连接 (边)
edges = [
    # 第一行横向、第二行横向、第三行横向
    [25, 31], [31, 38],
    [30, 37], [37, 43],
    [36, 42], [42, 49],
    # 第一列纵向、第二列纵向、第三列纵向
    [25, 30], [30, 36],
    [31, 37], [37, 42],
    [38, 43], [43, 49]
]

# 物理机的两比特门通常是双向的，所以我们要把单向边补全为双向边
coupling_list = []
for edge in edges:
    coupling_list.append(edge)
    coupling_list.append([edge[1], edge[0]]) # 反向连接

# 生成 Qiskit 可识别的拓扑对象
coupling_map = CouplingMap(couplinglist=coupling_list)

# 3. 将你的逻辑线路 (虚拟比特 0~8) 锁定到这些物理比特上
# 列表的第 i 个元素代表逻辑比特 i 映射到的物理比特编号
layout = physical_qubits 

# 4. 执行终极 Transpile 
transpiled_qc = transpile(
    qc, 
    basis_gates=['h', 'rx', 'ry', 'rz', 'cz'], 
    coupling_map=coupling_map,      # 告诉 Qiskit 硬件长什么样
    initial_layout=layout,          # 告诉 Qiskit 把逻辑比特放在哪
    optimization_level=3,           # 开启最高级别优化
    routing_method='sabre',         # Sabre 算法在处理网格拓扑的 SWAP 路由时效果最好
    layout_method='sabre'
)

# 打印一下深度和操作数，看看优化效果
print("编译后线路深度:", transpiled_qc.depth())
print("门数量统计:", transpiled_qc.count_ops())
qasm2.dumps(transpiled_qc)

编译后线路深度: 258
门数量统计: OrderedDict({'rx': 126, 'cz': 111, 'ry': 91, 'rz': 90, 'measure': 9, 'h': 3, 'barrier': 1})


'OPENQASM 2.0;\ninclude "qelib1.inc";\nqreg q[50];\ncreg meas[9];\nrz(-3*pi/4) q[25];\nry(pi/2) q[25];\nrx(pi/2) q[30];\nrx(pi/2) q[31];\nrz(-0.4579602204090283) q[36];\nry(1.5441183140448915) q[36];\nrz(-1.4633352156271409) q[36];\nrx(pi/2) q[37];\ncz q[30],q[37];\nrx(pi/2) q[30];\nrx(pi/2) q[37];\ncz q[30],q[37];\nrx(pi/2) q[30];\nrx(pi/2) q[37];\ncz q[30],q[37];\nrz(-0.8287484769708406) q[30];\nry(2.816142424699386) q[30];\nrz(1.1468921379170256) q[30];\ncz q[30],q[36];\nrx(1.4480331593863662) q[30];\nry(pi/2) q[36];\nrz(-3.0761116004064597) q[36];\ncz q[30],q[36];\nrz(pi/2) q[30];\nry(-0.7070849847493608) q[30];\nry(-pi/2) q[36];\nrz(0.45508096349320226) q[36];\ncz q[30],q[36];\nrz(-1.5651640278330776) q[30];\nry(1.7123411149019716) q[30];\nrz(0.16767962640120615) q[30];\nry(1.9428971371051322) q[36];\nrz(-0.3879951577154812) q[36];\nh q[37];\nrz(-1.7775883771929823) q[38];\nrx(-pi/2) q[38];\nrx(pi/2) q[42];\ncz q[42],q[36];\nrx(pi/2) q[36];\nrx(pi/2) q[42];\ncz q[42],q[36];\nrx(pi

In [20]:

# 使用夸父进行实验：
# transpiled_qc = transpile(qc, basis_gates=['h','rx', 'ry', 'rz', 'cz'], optimization_level=3)

qasm2.dump(transpiled_qc, open('../gates_2patterns/transpiled_circuit.qasm', 'w'))
